# HW3: Part II - Web Scraping Billboard Music Charts - SOLUTIONS

The Billboard Hot 100 is the music industry standard record chart in the United States for singles, published weekly by Billboard magazine. Chart rankings are based on sales (physical and digital), radio play, and online streaming.  These rankings are used to determine the top singles each year, which can be accessed on Wikipedia.

In this part of the homework you will perform some web scraping of this data. We introduced the Beautiful Soup library in class and how we can use it to extract data from HTML.  Recall that developer tools within Chrome (or simlar tools in other browsers) allows us to view the HTML that makes up a web page.

For additional reference on the Beautiful Soup Library, see Toolbox 11.1 on pg 519 in *Python for Everyone* (pg 643 in 2nd edition).

### Instructions:
* Type your **code** below each question that follows and show the output.  Some code is already provided for you.
* Use **markdown** cells to help organize and explain what you are doing.
* Rename this file by replacing the word "Blank" in the filename with your last name.  

## Part 1: Making a Beautiful Soup Object

We will use the year 1997 as an example of scraping data from the Billboard year-end hot 100 singles Wikipedia paage.

In [1]:
import requests
from bs4 import BeautifulSoup
from IPython.display import Image
BASE_URL = "https://en.wikipedia.org/wiki/Billboard_Year-End_Hot_100_singles_of_"

headers = {
            "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/115.0.0.0 Safari/537.36"
    ),
}

def get_soup(year):
    url = BASE_URL + str(year)
    page = requests.get(url,headers=headers)
    soup = BeautifulSoup(page.content, 'html.parser')
    return soup

year = 1997
soup = get_soup(year)
soup

<!DOCTYPE html>

<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available skin-thumbsize-clientpref-standard" dir="ltr" lang="en">
<head>
<meta charset="utf-8"/>
<title>Billboard Year-End Hot 100 singles of 1997 - Wikipedia</title>
<script>(function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-m

## Part 2: Store the chart data in a list of dictionaries

The end goal of this part is to make a list called `music_table`.  The list will contain 100 dictionaries, where each dictionary is of the form:

`{song: song_name, artist: [name1, name2, ...], artistURL: [url1, url2, ...], rank: int, year: int}`

That is, each row in the original music data chart will be convereted to a dictionary with five keys:
* The `song` key holds a string (song name),
* the `artist` key holds a list of strings (artist names), 
* the `artistURL` holds a list of strings (urls), 
* the `rank` key holds an int (rank of song), and 
* the `year` key holds the year as in int (in this case they will all be 1997).

In the end, you should have a list of dictionaries that looks like the below.

`[{'artist': ['Elton John'],
  'artistURL': ['/wiki/Elton_John'],
  'rank': 1,
  'song': '"Something About the Way You Look Tonight" / "Candle in the Wind 1997"',
  'year': 1997},
 {'artist': ['Jewel'],
  'artistURL': ['/wiki/Jewel_(singer)'],
  'rank': 2,
  'song': '"Foolish Games" / "You Were Meant for Me"',
  'year': 1997},
 {'artist': ['Puff Daddy', 'Faith Evans', '112'],
  'artistURL': ['/wiki/Sean_Combs', '/wiki/Faith_Evans', '/wiki/112_(band)'],
  'rank': 3,
  'song': '"I\'ll Be Missing You"',
  'year': 1997}
  ...]`

**Hints**:
* the chart data is in a **table** element with class "wikitable"
* each row of data is an html data row (think: "tr" html tag)
* within the row, the data you need for your dictionary can be found inside a **"td"** tag and an **"a"** tag within a **"td"** tag
* the `find_all` function is often useful on Beautiful Soup objects
* to extract urls, find `a`tags then extract using dictionary-indexing with key `href` or else use `.attrs['href']` on them
* if a song has multiple artists and/or links and you are having trouble recording each one into your dictionary, just record the first one in the list
* you might find it easier to break tasks up into functions in Part 2; then you can call them in Part 3; note there already is a get_soup function above that you can use
* occassionally, an artist will span 2 rows - see songs ranked 49 and 50 in 1997; you will need to write code to catch and address this; that row will have 2 items in it instead of 3 (rank and song title, but not artist)

### Some prep work

In [29]:
soup.find("table",class_="wikitable")

<table class="wikitable sortable" id="mwhw" style="text-align: center">
<caption id="mwiA">List of songs on <i id="mwiQ"><a href="https://en.wikipedia.org/wiki/Billboard_(magazine)" id="mwig" rel="mw:WikiLink" title="Billboard (magazine)">Billboard</a></i><span about="#mwt14" class="nowrap" data-mw="{&quot;parts&quot;:[{&quot;template&quot;:{&quot;target&quot;:{&quot;wt&quot;:&quot;'s&quot;,&quot;href&quot;:&quot;./Template:'s&quot;},&quot;params&quot;:{},&quot;i&quot;:0}}]}" id="mwiw" style="padding-left:0.1em;" typeof="mw:Transclusion"><span typeof="mw:Entity">'</span>s</span> 2021 <a href="https://en.wikipedia.org/wiki/Billboard_Year-End" id="mwjA" rel="mw:WikiLink" title="Billboard Year-End">Year-End</a> <a href="https://en.wikipedia.org/wiki/Billboard_Hot_100" id="mwjQ" rel="mw:WikiLink" title="Billboard Hot 100">Hot 100</a> chart<sup about="#mwt17" class="mw-ref reference" data-mw='{"name":"ref","attrs":{},"body":{"id":"mw-reference-text-cite_note-4"}}' id="cite_ref-4" rel="dc:re

In [2]:
rows = soup.find("table", class_="wikitable").find_all("tr")
sample_row = rows[1]
sample_row

<tr id="mwNg">
<td id="mwNw">1</td><td id="mwOA">"<a href="https://en.wikipedia.org/wiki/Candle_in_the_Wind_1997" id="mwOQ" rel="mw:WikiLink" title="Candle in the Wind 1997">Candle in the Wind 1997</a>" / "<a href="https://en.wikipedia.org/wiki/Something_About_the_Way_You_Look_Tonight" id="mwOg" rel="mw:WikiLink" title="Something About the Way You Look Tonight">Something About the Way You Look Tonight</a>"</td><td id="mwOw"><a href="https://en.wikipedia.org/wiki/Elton_John" id="mwPA" rel="mw:WikiLink" title="Elton John">Elton John</a></td></tr>

We can learn a lot from this sample row.  There are 3 entries - 1 per column, and they are rank, song title, and artist.
Entries 2 and 3 (song title and artist) have links.

In [3]:
# Scrape the content of each row
row_data = sample_row.find_all('td')
row_items = [item.get_text() for item in row_data]
row_items

['1',
 '"Candle in the Wind 1997" / "Something About the Way You Look Tonight"',
 'Elton John']

In [4]:
rank, song_title, artist_name = row_items
artist_name = artist_name.strip()   # there's a \n at its end we're removing
print(f'{song_title} by {artist_name} was ranked #{rank}.')

"Candle in the Wind 1997" / "Something About the Way You Look Tonight" by Elton John was ranked #1.


Next we will get the url links. Since occassionally there is more than one per artist and/or per song, we will store them in a list.

In [5]:
row_data[2]

<td id="mwOw"><a href="https://en.wikipedia.org/wiki/Elton_John" id="mwPA" rel="mw:WikiLink" title="Elton John">Elton John</a></td>

In [6]:
row_data[2].find('a')

<a href="https://en.wikipedia.org/wiki/Elton_John" id="mwPA" rel="mw:WikiLink" title="Elton John">Elton John</a>

Use .attrs to extract the url text (see hint above).

In [7]:
[link['href'] for link in row_data[1].find_all('a')]

['https://en.wikipedia.org/wiki/Candle_in_the_Wind_1997',
 'https://en.wikipedia.org/wiki/Something_About_the_Way_You_Look_Tonight']

In [8]:
[link['href'] for link in row_data[2].find_all('a')]

['https://en.wikipedia.org/wiki/Elton_John']

In [9]:
[link['href'] for link in row_data[0].find_all('a')]

[]

In [10]:
song_urls = [link['href'] for link in row_data[1].find_all('a')]
artist_urls = [link['href'] for link in row_data[2].find_all('a')]
song_urls,artist_urls

(['https://en.wikipedia.org/wiki/Candle_in_the_Wind_1997',
  'https://en.wikipedia.org/wiki/Something_About_the_Way_You_Look_Tonight'],
 ['https://en.wikipedia.org/wiki/Elton_John'])

### Putting it all together
This creates a list of dictionaries for a year's worth of Billboard rows.  You pass it the year's rows and year value.  
Note that as the hint points out, some artist titles span 2 rows, so there is some code to catch and handle that.

In [11]:
def make_music_table(rows, year):
    music_table = []
    for entry_row in rows[1:]:   # start at 1; 0th is header
        row_data = entry_row.find_all('td')
        row_items = [item.get_text() for item in row_data]
        rank, song_title = row_items[:2]
        song_title = song_title.strip()
        song_urls = [link['href'] for link in row_data[1].find_all('a')]

        if len(row_items) == 3:
            artist_name = row_items[2]
            artist_urls = [link.attrs['href'] for link in row_data[2].find_all('a')]
        else:  # tries to catch an artist title spanning 2 rows (see Celine Dion in 1997)
            print(f'Rank {rank}. {song_title} spans 2 rows')  # diagnostic
            # artist_name will use the previous one
            # artist_urls will use the previous one
            pass
        artist_name = artist_name.strip()   # there's a \n at its end we're removing
        
        entry_dict = {'year':year, 'rank':rank, 'song title':song_title, 'artist_name(s)':artist_name,
                     'song_url(s)':song_urls, 'artist_url(s)':artist_urls}
        music_table.append(entry_dict)

        #print(f'{song_title} by {artist_name} was ranked #{rank}.')
    return music_table
music_table = make_music_table(rows, year)

Rank 50. "It's All Coming Back to Me Now" spans 2 rows


Display the first few:

In [12]:
music_table[:3]

[{'year': 1997,
  'rank': '1',
  'song title': '"Candle in the Wind 1997" / "Something About the Way You Look Tonight"',
  'artist_name(s)': 'Elton John',
  'song_url(s)': ['https://en.wikipedia.org/wiki/Candle_in_the_Wind_1997',
   'https://en.wikipedia.org/wiki/Something_About_the_Way_You_Look_Tonight'],
  'artist_url(s)': ['https://en.wikipedia.org/wiki/Elton_John']},
 {'year': 1997,
  'rank': '2',
  'song title': '"You Were Meant for Me" / "Foolish Games"',
  'artist_name(s)': 'Jewel',
  'song_url(s)': ['https://en.wikipedia.org/wiki/You_Were_Meant_for_Me_(Jewel_song)',
   'https://en.wikipedia.org/wiki/Foolish_Games'],
  'artist_url(s)': ['https://en.wikipedia.org/wiki/Jewel_(singer)']},
 {'year': 1997,
  'rank': '3',
  'song title': '"I\'ll Be Missing You"',
  'artist_name(s)': 'Puff Daddy and Faith Evans featuring 112',
  'song_url(s)': ["https://en.wikipedia.org/wiki/I'll_Be_Missing_You"],
  'artist_url(s)': ['https://en.wikipedia.org/wiki/Sean_Combs',
   'https://en.wikipedia.

and the last few entries:

In [13]:
music_table[-3:]

[{'year': 1997,
  'rank': '98',
  'song title': '"You\'re Makin\' Me High" / "Let It Flow"',
  'artist_name(s)': 'Toni Braxton',
  'song_url(s)': ["https://en.wikipedia.org/wiki/You're_Makin'_Me_High",
   'https://en.wikipedia.org/wiki/Let_It_Flow_(song)'],
  'artist_url(s)': ['https://en.wikipedia.org/wiki/Toni_Braxton']},
 {'year': 1997,
  'rank': '99',
  'song title': '"You Must Love Me"',
  'artist_name(s)': 'Madonna',
  'song_url(s)': ['https://en.wikipedia.org/wiki/You_Must_Love_Me'],
  'artist_url(s)': ['https://en.wikipedia.org/wiki/Madonna']},
 {'year': 1997,
  'rank': '100',
  'song title': '"Let It Go"',
  'artist_name(s)': 'Ray J',
  'song_url(s)': ['https://en.wikipedia.org/wiki/Let_It_Go_(Ray_J_song)'],
  'artist_url(s)': ['https://en.wikipedia.org/wiki/Ray_J']}]

In [14]:
import pandas as pd

In [15]:
# not required but let's see it as a data frame
music_df = pd.DataFrame(music_table)
music_df.head()

,year,rank,song title,artist_name(s),song_url(s),artist_url(s)
0,1997,1,"""Candle in the Wind 1997"" / ""Something About t...",Elton John,[https://en.wikipedia.org/wiki/Candle_in_the_W...,[https://en.wikipedia.org/wiki/Elton_John]
1,1997,2,"""You Were Meant for Me"" / ""Foolish Games""",Jewel,[https://en.wikipedia.org/wiki/You_Were_Meant_...,[https://en.wikipedia.org/wiki/Jewel_(singer)]
2,1997,3,"""I'll Be Missing You""",Puff Daddy and Faith Evans featuring 112,[https://en.wikipedia.org/wiki/I'll_Be_Missing...,"[https://en.wikipedia.org/wiki/Sean_Combs, htt..."
3,1997,4,"""Un-Break My Heart""",Toni Braxton,[https://en.wikipedia.org/wiki/Un-Break_My_Heart],[https://en.wikipedia.org/wiki/Toni_Braxton]
4,1997,5,"""Can't Nobody Hold Me Down""",Puff Daddy featuring Mase,[https://en.wikipedia.org/wiki/Can't_Nobody_Ho...,"[https://en.wikipedia.org/wiki/Sean_Combs, htt..."


## Part 3:  Scraping data for 2002-2021

Once you have been able to accomplish the task of scraping from the web into a list of dictionaries for 1997, you will now do the same for 2002-2021.  You should realize that you can use much of the same code you used above, surrounded by a loop...the only thing changing is the year at the end of the webpage address.  So in general, the Billboard Year end Hot 100 singles Wikipedia page is

https://en.wikipedia.org/wiki/Billboard_Year-End_Hot_100_singles_of_####


You should end up with a list (call it `multi_yr_table`) that holds 2000 dictionaries, one for each row (and each artist) that appears in the Billboard Top 100 over the past 20 years.  We will connect the url to the year as we loop through the year. The last few entries are (song URLs aren't required):

` [{'year': 2021,
  'rank': '98\n',
  'song title': '"Tombstone"',
  'artist_name(s)': 'Rod Wave',
  'song_url(s)': ['/wiki/Tombstone_(song)'],
  'artist_url(s)': ['/wiki/Rod_Wave']},
 {'year': 2021,
  'rank': '99\n',
  'song title': '"Drinkin\' Beer. Talkin\' God. Amen."',
  'artist_name(s)': 'Chase Rice featuring Florida Georgia Line',
  'song_url(s)': ['/wiki/Drinkin%27_Beer._Talkin%27_God._Amen.'],
  'artist_url(s)': ['/wiki/Chase_Rice', '/wiki/Florida_Georgia_Line']},
 {'year': 2021,
  'rank': '100\n',
  'song title': '"Todo de Ti"',
  'artist_name(s)': 'Rauw Alejandro',
  'song_url(s)': ['/wiki/Todo_de_Ti'],
  'artist_url(s)': ['/wiki/Rauw_Alejandro']}]`

In [16]:
multi_yr_table = []
for year in range(2002,2022):
    print(year, end = ' ') # diagnostic
    page = requests.get(BASE_URL + str(year),headers=headers)
    soup = BeautifulSoup(page.content, 'html.parser')
    rows = soup.find("table", class_="wikitable").find_all("tr")
    music_table = make_music_table(rows,year)
    multi_yr_table.append(music_table)
# Let's see an entry
multi_yr_table[3][5]

2002 2003 2004 2005 2006 2007 2008 Rank 10. "Forever" spans 2 rows
Rank 17. "Don't Stop the Music" spans 2 rows
2009 2010 2011 2012 Rank 17. "Whistle" spans 2 rows
2013 Rank 18. "Wrecking Ball" spans 2 rows
2014 2015 Rank 10. "The Hills" spans 2 rows
2016 Rank 2. "Sorry" spans 2 rows
Rank 21. "Heathens" spans 2 rows
2017 2018 2019 2020 2021 

{'year': 2005,
 'rank': '6',
 'song title': '"Gold Digger"',
 'artist_name(s)': 'Kanye West featuring Jamie Foxx',
 'song_url(s)': ['https://en.wikipedia.org/wiki/Gold_Digger_(Kanye_West_song)'],
 'artist_url(s)': ['https://en.wikipedia.org/wiki/Kanye_West',
  'https://en.wikipedia.org/wiki/Jamie_Foxx']}

The last few:

In [17]:
multi_yr_table[-1][-3:]

[{'year': 2021,
  'rank': '98',
  'song title': '"Tombstone"',
  'artist_name(s)': 'Rod Wave',
  'song_url(s)': ['https://en.wikipedia.org/wiki/Tombstone_(Rod_Wave_song)'],
  'artist_url(s)': ['https://en.wikipedia.org/wiki/Rod_Wave']},
 {'year': 2021,
  'rank': '99',
  'song title': '"Drinkin\' Beer. Talkin\' God. Amen."',
  'artist_name(s)': 'Chase Rice featuring Florida Georgia Line',
  'song_url(s)': ["https://en.wikipedia.org/wiki/Drinkin'_Beer._Talkin'_God._Amen."],
  'artist_url(s)': ['https://en.wikipedia.org/wiki/Chase_Rice',
   'https://en.wikipedia.org/wiki/Florida_Georgia_Line']},
 {'year': 2021,
  'rank': '100',
  'song title': '"Todo de Ti"',
  'artist_name(s)': 'Rauw Alejandro',
  'song_url(s)': ['https://en.wikipedia.org/wiki/Todo_de_Ti'],
  'artist_url(s)': ['https://en.wikipedia.org/wiki/Rauw_Alejandro']}]

In [18]:
len(multi_yr_table)

20

In [19]:
# Let's create a flattened table to convert to a pandas data frame
flat_table = [item for yr_table in multi_yr_table for item in yr_table]

In [20]:
len(flat_table)

2000

In [21]:
flat_table[:3]

[{'year': 2002,
  'rank': '1',
  'song title': '"How You Remind Me"',
  'artist_name(s)': 'Nickelback',
  'song_url(s)': ['https://en.wikipedia.org/wiki/How_You_Remind_Me'],
  'artist_url(s)': ['https://en.wikipedia.org/wiki/Nickelback']},
 {'year': 2002,
  'rank': '2',
  'song title': '"Foolish"',
  'artist_name(s)': 'Ashanti',
  'song_url(s)': ['https://en.wikipedia.org/wiki/Foolish_(Ashanti_song)'],
  'artist_url(s)': ['https://en.wikipedia.org/wiki/Ashanti_(singer)']},
 {'year': 2002,
  'rank': '3',
  'song title': '"Hot in Herre"',
  'artist_name(s)': 'Nelly',
  'song_url(s)': ['https://en.wikipedia.org/wiki/Hot_in_Herre'],
  'artist_url(s)': ['https://en.wikipedia.org/wiki/Nelly']}]

In [22]:
flat_table[-3:]

[{'year': 2021,
  'rank': '98',
  'song title': '"Tombstone"',
  'artist_name(s)': 'Rod Wave',
  'song_url(s)': ['https://en.wikipedia.org/wiki/Tombstone_(Rod_Wave_song)'],
  'artist_url(s)': ['https://en.wikipedia.org/wiki/Rod_Wave']},
 {'year': 2021,
  'rank': '99',
  'song title': '"Drinkin\' Beer. Talkin\' God. Amen."',
  'artist_name(s)': 'Chase Rice featuring Florida Georgia Line',
  'song_url(s)': ["https://en.wikipedia.org/wiki/Drinkin'_Beer._Talkin'_God._Amen."],
  'artist_url(s)': ['https://en.wikipedia.org/wiki/Chase_Rice',
   'https://en.wikipedia.org/wiki/Florida_Georgia_Line']},
 {'year': 2021,
  'rank': '100',
  'song title': '"Todo de Ti"',
  'artist_name(s)': 'Rauw Alejandro',
  'song_url(s)': ['https://en.wikipedia.org/wiki/Todo_de_Ti'],
  'artist_url(s)': ['https://en.wikipedia.org/wiki/Rauw_Alejandro']}]

In [23]:
len(flat_table)

2000

Finally, convert this table to a pandas dataframe.
You don't need to do any analysis but note that if you've done this correctly, you will have scraped 2000 records from 20 different web pages into one dataframe to which  all of the pandas tools and techniques we have learned can be applied.

In [24]:
multi_yr_df = pd.DataFrame(flat_table)

multi_yr_df

,year,rank,song title,artist_name(s),song_url(s),artist_url(s)
0,2002,1,"""How You Remind Me""",Nickelback,[https://en.wikipedia.org/wiki/How_You_Remind_Me],[https://en.wikipedia.org/wiki/Nickelback]
1,2002,2,"""Foolish""",Ashanti,[https://en.wikipedia.org/wiki/Foolish_(Ashant...,[https://en.wikipedia.org/wiki/Ashanti_(singer)]
2,2002,3,"""Hot in Herre""",Nelly,[https://en.wikipedia.org/wiki/Hot_in_Herre],[https://en.wikipedia.org/wiki/Nelly]
3,2002,4,"""Dilemma""",Nelly featuring Kelly Rowland,[https://en.wikipedia.org/wiki/Dilemma_(Nelly_...,"[https://en.wikipedia.org/wiki/Nelly, https://..."
4,2002,5,"""Wherever You Will Go""",The Calling,[https://en.wikipedia.org/wiki/Wherever_You_Wi...,[https://en.wikipedia.org/wiki/The_Calling_(ba...
...,...,...,...,...,...,...
1995,2021,96,"""Things a Man Oughta Know""",Lainey Wilson,[https://en.wikipedia.org/wiki/Things_a_Man_Ou...,[https://en.wikipedia.org/wiki/Lainey_Wilson]
1996,2021,97,"""Throat Baby (Go Baby)""",BRS Kash,[https://en.wikipedia.org/wiki/Throat_Baby_(Go...,[https://en.wikipedia.org/wiki/BRS_Kash]
1997,2021,98,"""Tombstone""",Rod Wave,[https://en.wikipedia.org/wiki/Tombstone_(Rod_...,[https://en.wikipedia.org/wiki/Rod_Wave]
1998,2021,99,"""Drinkin' Beer. Talkin' God. Amen.""",Chase Rice featuring Florida Georgia Line,[https://en.wikipedia.org/wiki/Drinkin'_Beer._...,"[https://en.wikipedia.org/wiki/Chase_Rice, htt..."


In [25]:
# not required...
multi_yr_df['artist_name(s)'].value_counts()

artist_name(s)
Taylor Swift                                 24
Rihanna                                      21
Drake                                        19
Kelly Clarkson                               17
Pink                                         15
                                             ..
Cole Swindell                                 1
Lainey Wilson                                 1
BRS Kash                                      1
Chase Rice featuring Florida Georgia Line     1
Rauw Alejandro                                1
Name: count, Length: 1141, dtype: int64